# CosmoCalc: Python Calculator for Cosmogenic Nuclide Exposure Ages

**Authors:** Michal Ben-Israel  
**Based on:** Balco et al. (2008) CRONUS-Earth calculator v3  
**License:** GNU GPL v2  

## Overview
Python implementation of 10Be (and 26Al) surface exposure age and denudation rate 
calculator. Translated and updated from the MATLAB-based CRONUS-Earth framework 
(Balco et al., 2008) with the following updates:
- LSDn scaling scheme (Lifton et al., 2014)
- Updated production rates (Borchers et al., 2016)
- Updated muon production (Balco, 2017)

## References
- Balco et al. (2008) Quaternary Geochronology 3, 174-195
- Lifton et al. (2014) EPSL 386, 149-160
- Borchers et al. (2016) Quaternary Geochronology 31, 188-198
- Balco (2017) Quaternary Geochronology 39, 150-173

## 1. Load data files

In [28]:
import numpy as np
import scipy.io as sio
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator, interp1d
from scipy.integrate import cumulative_trapezoid

In [29]:
# ================================================
# CONSTANTS
# Borchers et al. (2016) Table 7
# SLHL spallation production rates (atoms/g quartz/yr)
# ================================================

# 10Be production rates
P10_St   = 4.01   # Stone (2000)
P10_Lm   = 4.00   # Lal/Stone paleomagnetic
P10_LSDn = 3.92   # Lifton-Sato-Dunai (preferred, v3 default)

# 26Al production rates
P26_St   = 27.93
P26_Lm   = 27.93
P26_LSDn = 28.54

# Decay constants (yr-1)
l10 = 4.9975e-7   # 10Be, half-life 1.387 Ma (Korschinek et al. 2010)
l26 = 9.83e-7     # 26Al, half-life 0.705 Ma (Nishiizumi 2004)

# Attenuation lengths (g/cm2)
Lsp  = 160.0      # spallation (Gosse & Phillips 2001)
Lmu  = 1500.0     # muons

## 2. Load data files

In [ ]:
# Load data files
ncep      = sio.loadmat('../data/NCEP2.mat',            squeeze_me=True, struct_as_record=False)
pmag07    = sio.loadmat('../data/PMag_Mar07.mat',        squeeze_me=True, struct_as_record=False)
pmag12    = sio.loadmat('../data/Pmag_Sep12.mat',        squeeze_me=True, struct_as_record=False)
lsd       = sio.loadmat('../data/consts_LSD.mat',        squeeze_me=True, struct_as_record=False)
consts_v23 = sio.loadmat('../data/al_be_consts_v23.mat', squeeze_me=True, struct_as_record=False)


## 3. Atmosphere
Converts sample elevation to atmospheric pressure using the NCEP reanalysis 
gridded sea-level pressure and temperature data (NCEPatm_2.m equivalent).

In [19]:
def ncep_pressure(lat, lon, elv):
    """
    Atmospheric pressure from elevation using NCEP reanalysis.
    Equivalent to NCEPatm_2.m (Balco v2.3).

    Inputs:
        lat : latitude (degrees N)
        lon : longitude (degrees E, 0-360)
        elv : elevation (m)
    Returns:
        pressure : atmospheric pressure (hPa)
    """
    NCEPlat   = ncep['NCEPlat']
    NCEPlon   = ncep['NCEPlon']
    meanslp   = ncep['meanslp']
    meant1000 = ncep['meant1000']

    interp_slp = RegularGridInterpolator((NCEPlat, NCEPlon), meanslp)
    interp_T   = RegularGridInterpolator((NCEPlat, NCEPlon), meant1000)

    slp = interp_slp([[lat, lon]])[0]
    T   = interp_T([[lat, lon]])[0] + 273.15

    gmr  = -0.03417
    dtdz =  0.0065

    pressure = slp * np.exp((gmr / dtdz) * (np.log(T) - np.log(T - elv * dtdz)))
    return pressure

# Test
print(f"Pressure at 37N, 240E, 3000m: {ncep_pressure(37.0, 240.0, 3000.0):.2f} hPa")

## 4. Load Sample Data
Reads sample input from CosmoCalc_in.xlsx.
Converts longitude to 0-360 for NCEP interpolation.
Applies AMS standard normalization to 07KNSTD.

In [40]:
def load_samples(filepath, sheet='CosmoCalc_in'):
    """
    Load sample data from CosmoCalc input spreadsheet.
    
    Inputs:
        filepath : path to Excel file
        sheet    : sheet name (default 'CosmoCalc_in')
    Returns:
        df : DataFrame with all sample data, 
             lon converted to 0-360,
             N10/delN10 normalized to 07KNSTD
    """
    df = pd.read_excel(filepath, sheet_name=sheet)
    
    # Convert longitude to 0-360 for NCEP interpolation
    df['lon_360'] = df['lon'] % 360
    
    # AMS standard normalization to 07KNSTD
    std_factors = {
        '07KNSTD'   : 1.0000,
        'KNSTD'     : 0.9996,
        'NIST_27900': 0.9785,
        'NIST_30000': 1.0000,
        'LLNL31000' : 1.0881,
        'LLNL10000' : 0.3511,
        'UVM-A'     : 0.0502,  # Corbett et al. 2019
        'BEST433'   : 0.9124,
        'S555'      : 0.9124,
    }
    
    df['std_factor'] = df['std'].map(std_factors)
    df['N10_norm']    = df['N10']    * df['std_factor']
    df['delN10_norm'] = df['delN10'] * df['std_factor']

## 5. Snow data (SNOTEL)
Snow shielding calculations based on Bason on Gosse & Phillips (2001)
Source: NRCS SNOTEL North Lake station (NTH), CA
Elevation: 9280 ft | Period of record: 1930-2026
Monthly SWE (snow water equivalent, cm)

In [41]:
df_snow = pd.read_csv('../data/NTH_snow.csv', skiprows=62, header=[0,1])

swe_cols = [('Jan', 'Snow Water Equivalent (in) Start of Month Values'),
            ('Feb', 'Snow Water Equivalent (in) Start of Month Values'),
            ('Mar', 'Snow Water Equivalent (in) Start of Month Values'),
            ('Apr', 'Snow Water Equivalent (in) Start of Month Values'),
            ('May', 'Snow Water Equivalent (in) Start of Month Values'),
            ('Jun', 'Snow Water Equivalent (in) Start of Month Values')]

swe_cm = np.zeros(12)
for i, col in enumerate(swe_cols):
    swe = pd.to_numeric(df_snow[col], errors='coerce')
    valid = swe.notna() & (swe > 0)
    if valid.sum() > 0:
        swe_cm[i] = swe[valid].mean() * 2.54

## 6. Cover Corrections
Calculates the scaling factor for sample thickness and surface cover 
(soil, till). Equivalent to the thickness correction in Balco v2.3,
extended to include multiple cover layers.

In [42]:
def cover_correction(rho_rock, thickness,
                     depth_soil=0.0, rho_soil=1.6,
                     depth_till=0.0, rho_till=1.9,
                     swe_monthly=None,
                     L=160.0):
    """
    Thickness and cover scaling factor for n samples.
    Vectorized — accepts scalars or arrays of length n.

    Inputs:
        rho_rock    : rock density (g/cm3), scalar or array
        thickness   : sample thickness (cm), scalar or array
        depth_soil  : soil depth (cm), scalar or array, default 0
        rho_soil    : soil density (g/cm3), default 1.6
        depth_till  : till depth (cm), scalar or array, default 0
        rho_till    : till density (g/cm3), default 1.9
        swe_monthly : array (12,) of monthly snow water equivalent (cm)
                      from SNOTEL or similar. None = no snow correction.
        L           : spallation attenuation length (g/cm2), default 160

    Returns:
        SF_thickness : rock+soil+till scaling factor, array length n
        SF_snow      : snow shielding factor, array length n
    """
    rho_rock   = np.atleast_1d(np.array(rho_rock,   dtype=float))
    thickness  = np.atleast_1d(np.array(thickness,  dtype=float))
    depth_soil = np.atleast_1d(np.array(depth_soil, dtype=float))
    depth_till = np.atleast_1d(np.array(depth_till, dtype=float))
    n = len(rho_rock)

    # Solid cover
    total_solid  = (rho_rock * thickness +
                    rho_soil * depth_soil +
                    rho_till * depth_till)
    SF_thickness = (L / total_solid) * (1 - np.exp(-total_solid / L))

    # Snow shielding via SWE (Gosse & Phillips 2001, Schildgen et al. 2005)
    if swe_monthly is None:
        SF_snow = np.ones(n)
    else:
        swe = np.array(swe_monthly, dtype=float)
        SF_snow = np.full(n, np.mean(np.exp(-swe / L)))

    return SF_thickness, SF_snow

## 7. Stone 2000 Scaling (St)
Time-independent scaling scheme after Lal (1991) as reformulated by Stone (2000).
Scales spallation production rate as a function of atmospheric pressure and latitude.
Equivalent to stone2000.m from Balco v2.3.
Reference: Stone (2000) JGR 105, 23753-23759.

In [43]:
def stone2000(lat, pressure, Fsp=1.0):
    """
    Stone (2000) scaling factor for cosmogenic nuclide production.
    Exact translation of stone2000.m from Balco v2.3.
    
    Inputs:
        lat      : latitude (degrees, scalar or array)
        pressure : atmospheric pressure (hPa, scalar or array)
        Fsp      : fraction of production by spallation (default 1.0)
                   Use 0.978 for 10Be, 0.974 for 26Al (Stone 2000 defaults)
    Returns:
        SF : scaling factor array
    """
    lat      = np.abs(np.atleast_1d(np.array(lat,      dtype=float)))
    pressure = np.atleast_1d(np.array(pressure, dtype=float))
    
    assert len(lat) == len(pressure), "lat and pressure must be same length"

    # Table 1 coefficients from Stone (2000)
    a = np.array([31.8518, 34.3699, 40.3153, 42.0983, 56.7733, 69.0720, 71.8733])
    b = np.array([250.3193, 258.4759, 308.9894, 512.6857, 649.1343, 832.4566, 863.1927])
    c = np.array([-0.083393, -0.089807, -0.106248, -0.120551, -0.160859, -0.199252, -0.207069])
    d = np.array([7.4260e-5, 7.9457e-5, 9.4508e-5, 1.1752e-4, 1.5463e-4, 1.9391e-4, 2.0127e-4])
    e = np.array([-2.2397e-8, -2.3697e-8, -2.8234e-8, -3.8809e-8, -5.0330e-8, -6.3653e-8, -6.6043e-8])
    ilats = np.array([0, 10, 20, 30, 40, 50, 60])

    # Clip latitude at 60 (same as Balco)
    lat = np.clip(lat, 0, 60)

    # Muon latitude factors (Table 1, Mx,1013.25)
    mk = np.array([0.587, 0.600, 0.678, 0.833, 0.933, 1.000, 1.000])

    S = np.zeros(len(lat))
    M = np.zeros(len(lat))

    for i in range(len(lat)):
        P = pressure[i]
        
        # Evaluate spallation polynomial at all 7 index latitudes
        # then interpolate to sample latitude -- exactly as Balco does it
        Slats = a + b*np.exp(-P/150.0) + c*P + d*P**2 + e*P**3
        S[i] = interp1d(ilats, Slats, kind='linear')(lat[i])

        # Muon scaling (Stone 2000 Eq. 3)
        Mlats = mk * np.exp((1013.25 - P) / 242.0)
        M[i] = interp1d(ilats, Mlats, kind='linear')(lat[i])

    # Combine spallation and muon terms
    SF = Fsp * S + (1 - Fsp) * M

    return SF

In [45]:
# Apply atmosphere, cover, and snow corrections to all samples
pressure   = np.array([ncep_pressure(row.lat, row.lon_360, row.elv) 
                       for _, row in df.iterrows()])

SF_thick, SF_snow = cover_correction(
    rho_rock   = df['rho_smp'].values,
    thickness  = df['thickness'].values,
    depth_soil = df['cover'].values,
    rho_soil   = df['rho_cover'].values,
    swe_monthly = swe_cm
)

SF_stone = stone2000(np.abs(df['lat'].values), pressure)

## 9. Exposure Age Solver — Stone 2000 (St)
Simple exposure age calculation using the time-independent St scaling scheme.
Closed-form solution — no forward integration needed.
Zero erosion assumed (surface samples).

In [48]:
def age_st(N10, delN10, P_total, l10=l10, delP_frac=0.35/4.01):
    """
    Exposure age and uncertainties from 10Be using Stone 2000 (St).
    Closed-form solution, zero erosion.
    
    Inputs:
        N10       : 10Be concentration (atoms/g)
        delN10    : 1σ uncertainty on N10 (atoms/g)
        P_total   : total production rate at site (atoms/g/yr)
        l10       : decay constant (yr-1)
        delP_frac : fractional uncertainty on production rate
    Returns:
        t         : exposure age (yr)
        delt_int  : internal uncertainty - measurement only (yr)
        delt_ext  : external uncertainty - production rate only (yr)
        delt_tot  : total uncertainty - combined in quadrature (yr)
    """
    if N10 >= P_total / l10:
        return np.inf, np.inf, np.inf, np.inf

    t = (-1/l10) * np.log(1 - (N10 * l10 / P_total))

    # Partial derivatives
    dtdN = 1.0 / (P_total - N10 * l10)
    dtdP = -t / P_total

    # Three uncertainty components
    delt_int = np.abs(dtdN) * delN10
    delt_ext = np.abs(dtdP) * P_total * delP_frac
    delt_tot = np.sqrt(delt_int**2 + delt_ext**2)

    return t, delt_int, delt_ext, delt_tot


# Calculate St ages with all uncertainties
print(f"{'Sample':<20} {'Age (ka)':>9} {'±int':>7} {'±ext':>7} {'±tot':>7}")
print("-"*55)

for i, row in df.iterrows():
    P_total = P10_St * SF_thick[i] * SF_snow[i] * SF_stone[i] * row.shieldcorr
    t, dint, dext, dtot = age_st(row.N10_norm, row.delN10_norm, P_total)
    print(f"{row['sample']:<20} {t/1000:>9.2f} "
          f"{dint/1000:>7.2f} {dext/1000:>7.2f} {dtot/1000:>7.2f}")

Sample                Age (ka)    ±int    ±ext    ±tot
-------------------------------------------------------
22-MBI-008-SBL1b         14.48    0.27    1.26    1.29
22-MBI-010-SBL2          15.39    0.29    1.34    1.38
22-MBI-020-SBL5b         15.43    0.42    1.35    1.41
22-MBI-045-SBL9b         12.69    0.24    1.11    1.13
23-MBI-009-SBL11         14.59    0.24    1.27    1.30
23-MBI-010-SBL12         14.12    0.24    1.23    1.26
22-MBI-061-SHL1          15.45    0.30    1.35    1.38
22-MBI-069-SHL2          17.17    0.32    1.50    1.53
22-MBI-076-SHL3          14.23    0.26    1.24    1.27
22-MBI-084-SHL4          15.27    0.25    1.33    1.36
22-MBI-086-SHL6          15.85    0.29    1.38    1.41


In [54]:
# Build the time vector
tv = np.concatenate([
    np.arange(0, 6500+500, 500),      # 0-6500 in 500yr steps
    np.array([6900]),                   # one extra point
    np.arange(7500, 11500+1000, 1000), # 7500-11500 in 1000yr steps
    np.arange(12000, 800000+1000, 1000), # 12000-800000 in 1000yr steps
    np.logspace(np.log10(810000), 7, 200) # log-spaced to 10Ma
])

## 10. Lm Scaling (Paleomagnetically Corrected Lal/Stone)
Time-dependent scaling scheme using paleomagnetically corrected cutoff rigidity.
Equivalent to stone2000Rcsp.m from Balco v2.3.
Requires: PMag_Mar07.mat for Rc(t) time series.

In [55]:
def stone2000Rcsp(pressure, Rc):
    """
    Stone 2000 scaling using cutoff rigidity Rc instead of latitude.
    Equivalent to stone2000Rcsp.m from Balco v2.3.
    Time-dependent version for Lm scaling.
    
    Inputs:
        pressure : atmospheric pressure (hPa), scalar
        Rc       : cutoff rigidity (GV), scalar or array of time steps
    Returns:
        SF : scaling factor array (same length as Rc)
    """
    # Rc to latitude conversion (dipole approximation)
    # Rc = 14.9 * cos^4(lambda) → lambda = arccos((Rc/14.9)^0.25)
    Rc   = np.atleast_1d(np.array(Rc, dtype=float))
    
    # Clip Rc to valid range
    Rc = np.clip(Rc, 0, 14.9)
    
    # Convert Rc to effective latitude
    lat_eff = np.degrees(np.arccos(np.clip((Rc / 14.9)**0.25, 0, 1)))
    lat_eff = np.clip(lat_eff, 0, 60)

    # Stone 2000 coefficients
    a = np.array([31.8518, 34.3699, 40.3153, 42.0983, 56.7733, 69.0720, 71.8733])
    b = np.array([250.3193, 258.4759, 308.9894, 512.6857, 649.1343, 832.4566, 863.1927])
    c = np.array([-0.083393, -0.089807, -0.106248, -0.120551, -0.160859, -0.199252, -0.207069])
    d = np.array([7.4260e-5, 7.9457e-5, 9.4508e-5, 1.1752e-4, 1.5463e-4, 1.9391e-4, 2.0127e-4])
    e = np.array([-2.2397e-8, -2.3697e-8, -2.8234e-8, -3.8809e-8, -5.0330e-8, -6.3653e-8, -6.6043e-8])
    ilats = np.array([0, 10, 20, 30, 40, 50, 60])

    SF = np.zeros(len(Rc))
    for i in range(len(Rc)):
        Slats = a + b*np.exp(-pressure/150.0) + c*pressure + d*pressure**2 + e*pressure**3
        SF[i] = interp1d(ilats, Slats, kind='linear')(lat_eff[i])

    return SF

In [60]:
from scipy.interpolate import RegularGridInterpolator

def angdist_py(lat1, lon1, lat2, lon2):
    """Angular distance between two points on a sphere (degrees)."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return np.degrees(2 * np.arcsin(np.sqrt(a)))


def compute_Rc_lm(lat, lon, tv, pmag):
    """
    Compute cutoff rigidity time series Rc(t) for Lm scaling.
    Exact translation of LmRc construction in Balco's cosmo_engine.m
    
    Steps 1-15  (0-6900 yr): KC pole positions + MM0_KCL
    Steps 16+   (7500+ yr) : axial dipole + MM0
    """
    MM0_KCL    = pmag['MM0_KCL']
    lat_pp_KCL = pmag['lat_pp_KCL']
    lon_pp_KCL = pmag['lon_pp_KCL']
    t_M        = pmag['t_M']
    MM0        = pmag['MM0']

    lat_rd = np.radians(lat)
    LmRc   = np.zeros(len(tv))

    # Segment 1: steps 1-15 (tv[0:15] = 0 to 6900 yr)
    for i in range(15):
        lm_lat    = abs(90 - angdist_py(lat, lon,
                                         lat_pp_KCL[i],
                                         lon_pp_KCL[i]))
        lm_lat_rd = np.radians(lm_lat)
        LmRc[i]   = 14.9 * MM0_KCL[i] * (np.cos(abs(lm_lat_rd))**4)

    # Segment 2: steps 16+ (tv[15:] = 7500 yr onward)
    t_M_ext  = np.append(t_M[:-1], t_M[-2] + 1000)
    MM0_ext  = np.append(MM0[:-1], MM0[-2])

    tv_rest    = tv[15:]
    tv_clipped = np.clip(tv_rest, t_M_ext[0], t_M_ext[-1])
    MM0_interp = interp1d(t_M_ext, MM0_ext,
                          kind='linear',
                          bounds_error=False,
                          fill_value=(MM0_ext[0], MM0_ext[-1]))(tv_clipped)

    LmRc[15:] = 14.9 * MM0_interp * (np.cos(abs(lat_rd))**4)

    return LmRc

In [65]:
def age_lm(N10, delN10, pressure, LmRc, tv,
           SF_thick, SF_snow, shieldcorr,
           P10_Lm=P10_Lm, l10=l10, delP_frac=0.37/4.00):
    """
    Exposure age from 10Be using Lm scaling (paleomagnetically corrected).
    Forward integration + reverse interpolation.
    
    Inputs:
        N10        : measured 10Be concentration (atoms/g)
        delN10     : 1σ uncertainty (atoms/g)
        pressure   : site pressure (hPa)
        LmRc       : cutoff rigidity time series (GV), length = len(tv)
        tv         : time vector (yr)
        SF_thick   : thickness scaling factor
        SF_snow    : snow shielding factor
        shieldcorr : topographic shielding factor
        P10_Lm     : SLHL production rate for Lm (Borchers 2016)
        l10        : decay constant (yr-1)
        delP_frac  : fractional production rate uncertainty
    Returns:
        t          : exposure age (yr)
        delt_int   : internal uncertainty (yr)
        delt_ext   : external uncertainty (yr)
        delt_tot   : total uncertainty (yr)
    """
    # Production rate time series P(t)
    # stone2000Rcsp gives scaling factor at each time step
    SF_t = stone2000Rcsp(pressure, LmRc)  # shape: len(tv)
    
    # Surface production rate at each time step
    P_t = P10_Lm * SF_t * SF_thick * SF_snow * shieldcorr

    # Decay factor
    dcf = np.exp(-tv * l10)

    # Forward integrate N(t) = cumtrapz(P(t) * decay)
    N_t = cumulative_trapezoid(P_t * dcf, tv, initial=0)

    # Check saturation
    if N10 >= np.max(N_t):
        return np.inf, np.inf, np.inf, np.inf

    # Reverse interpolate to find age
    t = interp1d(N_t, tv, kind='linear')(N10)

    # Error propagation (same approach as St)
    # Effective production rate that would give this age in simple equation
    A = l10  # no erosion
    FP = (N10 * A) / (1 - np.exp(-A * t))

    dtdN = 1.0 / (FP - N10 * A)
    dtdP = -N10 / (FP**2 - N10 * A * FP)

    delt_int = np.abs(dtdN) * delN10
    delt_ext = np.abs(dtdP) * FP * delP_frac
    delt_tot = np.sqrt(delt_int**2 + delt_ext**2)

    return float(t), delt_int, delt_ext, delt_tot


# Calculate Lm ages for all samples
print(f"{'Sample':<20} {'Age_St(ka)':>11} {'Age_Lm(ka)':>11} {'±int':>7} {'±ext':>7} {'±tot':>7}")
print("-"*67)

for i, row in df.iterrows():
    # St age for comparison
    P_st = P10_St * SF_thick[i] * SF_snow[i] * SF_stone[i] * row.shieldcorr
    t_st, _, _, _ = age_st(row.N10_norm, row.delN10_norm, P_st)

    # Lm age
    LmRc_i = compute_Rc_lm(row.lat, row.lon_360, tv, pmag07)
    t_lm, dint, dext, dtot = age_lm(
        N10        = row.N10_norm,
        delN10     = row.delN10_norm,
        pressure   = pressure[i],
        LmRc       = LmRc_i,
        tv         = tv,
        SF_thick   = SF_thick[i],
        SF_snow    = SF_snow[i],
        shieldcorr = row.shieldcorr
    )
    print(f"{row['sample']:<20} {t_st/1000:>11.2f} {t_lm/1000:>11.2f} "
          f"{dint/1000:>7.2f} {dext/1000:>7.2f} {dtot/1000:>7.2f}")

Sample                Age_St(ka)  Age_Lm(ka)    ±int    ±ext    ±tot
-------------------------------------------------------------------
22-MBI-008-SBL1b           14.48       14.01    0.26    1.30    1.33
22-MBI-010-SBL2            15.39       14.87    0.28    1.38    1.41
22-MBI-020-SBL5b           15.43       14.91    0.41    1.38    1.44
22-MBI-045-SBL9b           12.69       12.28    0.23    1.14    1.16
23-MBI-009-SBL11           14.59       14.10    0.23    1.31    1.33
23-MBI-010-SBL12           14.12       13.65    0.23    1.27    1.29
22-MBI-061-SHL1            15.45       14.90    0.29    1.38    1.41
22-MBI-069-SHL2            17.17       16.48    0.31    1.53    1.56
22-MBI-076-SHL3            14.23       13.76    0.26    1.28    1.30
22-MBI-084-SHL4            15.27       14.74    0.24    1.37    1.39
22-MBI-086-SHL6            15.85       15.28    0.28    1.42    1.45


## 11. LSDn Scaling (Lifton-Sato-Dunai, nuclide specific)
Time-dependent scaling using Sato et al. (2008) cosmic ray flux model.
The preferred scaling scheme for v3 and what the reviewer used.
Reference: Lifton et al. (2014) EPSL 386, 149-160.

In [73]:
def build_lsdn_timevectors(lat, lon, pmag, consts):
    """
    Build time vector, Rc(t) and SPhi(t) for LSDn scaling.
    Translated from lsdrc() and solar_modulation() in TCNtools/R/scaling_lsd.R
    
    Inputs:
        lat    : sample latitude (degrees N)
        lon    : sample longitude (degrees E, 0-360)
        pmag   : loaded PMag_Sep12.mat dictionary
        consts : loaded consts_LSD.mat dictionary
    Returns:
        tv        : time vector (yr)
        LSDRc     : cutoff rigidity time series (GV)
        this_SPhi : solar modulation time series (MV)
    """
    c   = consts['consts']   # LSD constants
    t_M = pmag['t_M']        # dipole moment time vector (from PMag_Sep12)
    MM0 = pmag['MM0']        # relative dipole moment (from PMag_Sep12)

    # Build time vector - same as TCNtools
    tv = np.concatenate([
        np.arange(0, 51, 10),                  # 0-50 in 10yr steps (6 points)
        np.arange(60, 50061, 100),              # 60-50060 in 100yr steps (114 points)
        np.arange(51060, 2000061, 1000),        # 51060-2000060 in 1000yr steps
        np.logspace(np.log10(2001060), 7, 200) # log-spaced to 10Ma
    ])

    # Solar modulation SPhi(t)
    this_SPhi = np.full(len(tv), c.SPhiInf)
    this_SPhi[:120] = c.SPhi  # first 120 points use actual record

    # Rc(t) construction
    lon_360 = lon % 360
    lat_rd  = np.radians(lat)
    LSDRc   = np.zeros(len(tv))

    # Segment 1: first 76 steps from TTRc grid (0-6960 yr)
    interp_Rc = RegularGridInterpolator(
        (c.lat_Rc, c.lon_Rc, c.t_Rc),
        c.TTRc,
        method='linear', bounds_error=False, fill_value=None
    )
    tempRc = np.array([
        interp_Rc([[lat, lon_360, t]])[0] for t in c.t_Rc
    ])
    LSDRc[:76] = interp1d(c.t_Rc, tempRc, kind='linear',
                          bounds_error=False,
                          fill_value=(tempRc[0], tempRc[-1]))(tv[:76])

    # Segment 2: beyond 76 steps - 6th order polynomial
    # Trajectory-traced GAD dipole field from TCNtools lsdrc()
    dd = np.array([6.89901, -103.241, 522.061, -1152.15, 1189.18, -448.004])

    tv_rest    = tv[76:]
    MM0_interp = interp1d(t_M, MM0,
                          kind='linear',
                          bounds_error=False,
                          fill_value=(MM0[0], MM0[-1]))(
                              np.clip(tv_rest, t_M[0], t_M[-1]))

    cos_lat = np.cos(abs(lat_rd))
    Rc_geo  = (dd[0]*cos_lat   + dd[1]*cos_lat**2 + dd[2]*cos_lat**3 +
               dd[3]*cos_lat**4 + dd[4]*cos_lat**5 + dd[5]*cos_lat**6)

    LSDRc[76:] = MM0_interp * Rc_geo

    return tv, LSDRc, this_SPhi

In [78]:
def sato_neutrons_py(h, Rc, s, w, consts_lsd, xsects):
    """
    Neutron flux using Sato et al. (2008) PARMA model.
    Vectorized translation of sato_neutrons.R (TCNtools/Lifton 2014)
    
    Inputs:
        h    : atmospheric pressure (hPa), scalar
        Rc   : cutoff rigidity (GV), array length n
        s    : solar modulation (MV), array length n  
        w    : fractional water content (default 0.066)
        consts_lsd : loaded consts_LSD.mat
        xsects     : loaded XSectsReedyAll.mat
    Returns:
        P10n  : Be-10 neutron production rate (atoms/g/yr), array length n
        nflux : integral neutron flux, array length n
    """
    c  = consts_lsd['consts']
    xs = xsects

    x  = h * 1.019716          # pressure to atm depth (g/cm2)
    E  = np.logspace(0, 5.3010, 200)  # energy bins (MeV)

    Rc = np.array(Rc, dtype=float).copy()
    Rc[Rc < 1.0] = 1.0
    s  = np.array(s,  dtype=float)

    smin = 400.0
    smax = 1200.0

    # ---- b-coefficients (Table 1 of Sato 2008) ----
    b11min=2.5702e1;  b11max=-6.9221
    b12min=-5.0931e-1;b12max=1.1336
    b13min=7.4650;    b13max=2.6961e1
    b14min=1.2313e1;  b14max=1.1746e1
    b15min=1.0498;    b15max=2.6171
    b21min=6.5143e-3; b21max=5.3211e-3
    b22min=3.3511e-5; b22max=8.4899e-5
    b23min=9.4415e-4; b23max=2.0704e-3
    b24min=1.2088e1;  b24max=1.1714e1
    b25min=2.7782;    b25max=3.8051
    b31min=9.8364e-1; b31max=9.7536e-1
    b32min=1.4964e-4; b32max=6.4733e-4
    b33min=-7.3249e-1;b33max=-2.2750e-1
    b34min=-1.4381;   b34max=2.0713
    b35min=2.7448;    b35max=2.1689
    b41min=8.8681e-3; b41max=9.1435e-3
    b42min=-4.3322e-5;b42max=-6.4855e-5
    b43min=1.7293e-2; b43max=5.8179e-3
    b44min=-1.0836;   b44max=1.0168
    b45min=2.6602;    b45max=2.4504

    b121=9.31e-1; b122=3.70e-2;  b123=-2.02;   b124=2.12; b125=5.34
    b131=6.67e-4; b132=-1.19e-5; b133=1.00e-4; b134=1.45; b135=4.29

    b51=9.7337e-4; b52=-9.6635e-5; b53=1.2121e-2; b54=7.1726; b55=1.4601
    b91=5.7199e2;  b92=7.1293;     b93=-1.0703e2; b94=1.8538; b95=1.2142
    b101=6.8552e-4;b102=2.7136e-5; b103=5.7823e-4;b104=8.8534;b105=3.6417
    b111=-5.0800e-1;b112=1.4783e-1;b113=1.0068;   b114=9.1556;b115=1.6369

    a6=1.8882e-4; a7=4.4791e-1; a8=1.4361e-3; a12=1.4109e-2

    c1=2.3555e-1; c2=2.3779;    c3=7.2597e-1
    c5=1.2391e2;  c6=2.2318;    c7=1.0791e-3
    c8=3.6435e-12;c9=1.6595;    c10=8.4782e-8; c11_=1.5054

    # Ground-level spectrum modifier (water content dependent, E only)
    h31=-2.5184e1; h32=2.7298; h33=7.1526e-2
    h51=3.4790e-1; h52=3.3493; h53=-1.5744
    h61=1.1800e-1; h62=1.4438e-1; h63=3.8733
    h64=6.5298e-1; h65=4.2752e1

    g1_=-0.023499; g2_=-0.012938
    g3_ = 10**(h31 + h32/(w + h33))
    g4_ = 9.6889e-1
    g5_w = h51 + h52*w + h53*(w**2)

    fG  = 10**(g1_ + g2_*np.log10(E/g3_)*(1 - np.tanh(g4_*np.log10(E/g5_w))))

    Et  = 2.5e-8
    g6_ = (h61 + h62*np.exp(-h63*w))/(1 + h64*np.exp(-h65*w))
    PhiT = g6_*((E/Et)**2)*np.exp(-E/Et)

    # ---- Rc-dependent scalars (shape n,) ----
    a1min = b11min + b12min*Rc + b13min/(1+np.exp((Rc-b14min)/b15min))
    a1max = b11max + b12max*Rc + b13max/(1+np.exp((Rc-b14max)/b15max))
    a2min = b21min + b22min*Rc + b23min/(1+np.exp((Rc-b24min)/b25min))
    a2max = b21max + b22max*Rc + b23max/(1+np.exp((Rc-b24max)/b25max))
    a3min = b31min + b32min*Rc + b33min/(1+np.exp((Rc-b34min)/b35min))
    a3max = b31max + b32max*Rc + b33max/(1+np.exp((Rc-b34max)/b35max))
    a4min = b41min + b42min*Rc + b43min/(1+np.exp((Rc-b44min)/b45min))
    a4max = b41max + b42max*Rc + b43max/(1+np.exp((Rc-b44max)/b45max))

    a5_  = b51  + b52*Rc  + b53/(1+np.exp((Rc-b54)/b55))
    a9_  = b91  + b92*Rc  + b93/(1+np.exp((Rc-b94)/b95))
    a10_ = b101 + b102*Rc + b103/(1+np.exp((Rc-b104)/b105))
    a11_ = b111 + b112*Rc + b113/(1+np.exp((Rc-b114)/b115))
    b5_  = b121 + b122*Rc + b123/(1+np.exp((Rc-b124)/b125))
    b6_  = b131 + b132*Rc + b133/(1+np.exp((Rc-b134)/b135))

    c4_  = a5_ + a6*x/(1 + a7*np.exp(a8*x))           # (n,)
    c12_ = a9_*(np.exp(-a10_*x) + a11_*np.exp(-a12*x)) # (n,)

    PhiLmin = a1min*(np.exp(-a2min*x) - a3min*np.exp(-a4min*x))
    PhiLmax = a1max*(np.exp(-a2max*x) - a3max*np.exp(-a4max*x))

    f3_ = b5_ + b6_*x
    f2_ = (PhiLmin - PhiLmax)/(smin**f3_ - smax**f3_)
    f1_ = PhiLmin - f2_*smin**f3_
    PhiL = f1_ + f2_*s**f3_   # (n,)

    # ---- Broadcasting: (n,1) × (1,200) → (n,200) ----
    E_2d    = E[np.newaxis,:]       # (1,200)
    c4_2d   = c4_[:,np.newaxis]     # (n,1)
    c12_2d  = c12_[:,np.newaxis]    # (n,1)
    PhiL_2d = PhiL[:,np.newaxis]    # (n,1)

    PhiB = ((c1*(E_2d/c2)**c3)*np.exp(-E_2d/c2) +
            c4_2d*np.exp(-(np.log10(E_2d)-np.log10(c5))**2/(2*np.log10(c6)**2)) +
            c7*np.log10(E_2d/c8)*
            (1+np.tanh(c9*np.log10(E_2d/c10)))*
            (1-np.tanh(c11_*np.log10(E_2d/c12_2d))))  # (n,200)

    PhiG    = PhiL_2d*(PhiB*fG + PhiT)   # (n,200)
    PhiGMev = PhiG / E_2d                 # (n,200)

    # ---- Production rates ----
    Natoms10 = float(c.Natoms10)
    conv     = 1e-27 * 3.1536e7

    P10n = (np.trapezoid(PhiGMev * xs['O16nxBe10'][np.newaxis,:], E, axis=1) +
            np.trapezoid(PhiGMev * xs['SinxBe10'][np.newaxis,:]/2, E, axis=1)) * Natoms10 * conv

    nflux = np.trapezoid(PhiGMev, E, axis=1)

    return P10n, nflux


def sato_protons_py(h, Rc, s, consts_lsd, xsects):
    """
    Proton flux using Sato et al. (2008) PARMA model.
    Vectorized translation of sato_protons.R (TCNtools/Lifton 2014)
    """
    c  = consts_lsd['consts']
    xs = xsects

    x  = h * 1.019716
    E  = np.logspace(0, 5.3010, 200)

    Rc = np.array(Rc, dtype=float).copy()
    Rc[Rc < 1.0] = 1.0
    s  = np.array(s, dtype=float)

    smin=400.0; smax=1200.0
    A=1.0; Z=1.0; Ep=938.27
    U=(4-1.675)*np.pi*A/Z*1e-7

    a1=2.1153; a2=4.4511e-1; a3=1.0064e-2; a4=3.9564e-2
    a5=2.9236; a6=2.7076
    a7=1.2663e4; a8=4.8288e3; a9=3.2822e4; a10=7.4378e3
    a11=3.4643; a12=1.6752; a13=1.3691; a14=2.0665
    a15=1.0833e2; a16=2.3013e3

    Etoa = E + a1*x
    Rtoa = 0.001*np.sqrt((A*Etoa)**2 + 2*A*Ep*Etoa)/Z

    c11=1.2560;  c12=3.2260e-3; c13=-4.8077e-6; c14=2.2825e-9
    c21=4.3783e-1;c22=-5.5759e-4;c23=7.8388e-7; c24=-3.8671e-10
    c31=1.8102e-4;c32=-5.1754e-7;c33=7.5876e-10;c34=-3.8220e-13
    c41=1.7065;  c42=7.1608e-4; c43=-9.3220e-7; c44=5.2665e-10

    b1=c11+c12*x+c13*x**2+c14*x**3
    b2=c21+c22*x+c23*x**2+c24*x**3
    b3=c31+c32*x+c33*x**2+c34*x**3
    b4=c41+c42*x+c43*x**2+c44*x**3

    h11min=2.4354e-3; h11max=2.5450e-3
    h12min=-6.0339e-5;h12max=-7.1807e-5
    h13min=2.1951e-3; h13max=1.4580e-3
    h14min=6.6767;    h14max=6.9150
    h15min=9.3228e-1; h15max=9.9366e-1
    h21min=7.7872e-3; h21max=7.6828e-3
    h22min=-9.5771e-6;h22max=-2.4119e-6
    h23min=6.2229e-4; h23max=6.6411e-4
    h24min=7.7842;    h24max=7.7461
    h25min=1.8502;    h25max=1.9431
    h31min=9.6340e-1; h31max=9.7353e-1
    h32min=1.5974e-3; h32max=1.0577e-3
    h33min=-7.1179e-2;h33max=-2.1383e-2
    h34min=2.2320;    h34max=3.0058
    h35min=7.8800e-1; h35max=9.1845e-1
    h41min=7.8132e-3; h41max=7.3482e-3
    h42min=9.7085e-11;h42max=2.5598e-5
    h43min=8.2392e-4; h43max=1.2457e-3
    h44min=8.5138;    h44max=8.1896
    h45min=2.3125;    h45max=2.9368
    h51=1.9100e-1; h52=7.0300e-2; h53=-6.4500e-1; h54=2.0300; h55=1.3000
    h61=5.7100e-4; h62=6.1300e-6; h63=5.4700e-4; h64=1.1100; h65=8.3700e-1

    g1min=h11min+h12min*Rc+h13min/(1+np.exp((Rc-h14min)/h15min))
    g1max=h11max+h12max*Rc+h13max/(1+np.exp((Rc-h14max)/h15max))
    g2min=h21min+h22min*Rc+h23min/(1+np.exp((Rc-h24min)/h25min))
    g2max=h21max+h22max*Rc+h23max/(1+np.exp((Rc-h24max)/h25max))
    g3min=h31min+h32min*Rc+h33min/(1+np.exp((Rc-h34min)/h35min))
    g3max=h31max+h32max*Rc+h33max/(1+np.exp((Rc-h34max)/h35max))
    g4min=h41min+h42min*Rc+h43min/(1+np.exp((Rc-h44min)/h45min))
    g4max=h41max+h42max*Rc+h43max/(1+np.exp((Rc-h44max)/h45max))

    phiPmin=g1min*(np.exp(-g2min*x)-g3min*np.exp(-g4min*x))
    phiPmax=g1max*(np.exp(-g2max*x)-g3max*np.exp(-g4max*x))

    g5_=h51+h52*Rc+h53/(1+np.exp((Rc-h54)/h55))
    g6_=h61+h62*Rc+h63/(1+np.exp((Rc-h64)/h65))

    f3_=g5_+g6_*x
    f2_=(phiPmin-phiPmax)/(smin**f3_-smax**f3_)
    f1_=phiPmin-f2_*smin**f3_
    phiP=f1_+f2_*s**f3_   # (n,)

    # Broadcasting (n,200)
    E_2d    = E[np.newaxis,:]
    s_2d    = s[:,np.newaxis]
    phiP_2d = phiP[:,np.newaxis]

    Elis_2d = Etoa[np.newaxis,:] + s_2d*Z/A
    Beta_2d = np.sqrt(1-(Ep/(Ep+Elis_2d*A))**2)
    Rlis_2d = 0.001*np.sqrt((A*Elis_2d)**2+2*A*Ep*Elis_2d)/Z
    C_2d    = a7+a8/(1+np.exp((Elis_2d-a9)/a10))

    phiTOA_2d = (C_2d*(Beta_2d**a5)/(Rlis_2d**a6))*(Rtoa[np.newaxis,:]/Rlis_2d)**2
    phiPri_2d = (U/Beta_2d)*phiTOA_2d*(a2*np.exp(-a3*x)+(1-a2)*np.exp(-a4*x))
    phiSec_2d = (phiP_2d*b1*E_2d**b2)/(1+b3*E_2d**b4)

    Ec_  = (np.sqrt((1000*Rc*Z)**2+Ep**2)-Ep)/A
    Es_  = a13*(Ec_-a14*x)
    Es1_ = np.maximum(a15, Es_)[:,np.newaxis]
    Es2_ = np.maximum(a16, Es_)[:,np.newaxis]

    phiPtot = (phiPri_2d*(np.tanh(a11*(E_2d/Es1_-1))+1)/2 +
               phiSec_2d*(np.tanh(a12*(1-E_2d/Es2_))+1)/2)  # (n,200)

    Natoms10 = float(c.Natoms10)
    conv     = 1e-27*3.1536e7

    P10p = (np.trapezoid(phiPtot*xs['O16pxBe10'][np.newaxis,:], E, axis=1) +
            np.trapezoid(phiPtot*xs['SipxBe10'][np.newaxis,:]/2, E, axis=1))*Natoms10*conv

    pflux = np.trapezoid(phiPtot, E, axis=1)

    return P10p, pflux

## 12. LSDn Age Solver (Lifton-Sato-Dunai, nuclide specific)

The LSDn scaling scheme uses the Sato et al. (2008) PARMA analytical model 
to compute the cosmic ray flux (neutrons + protons) at any site and time, 
given atmospheric pressure, cutoff rigidity Rc(t), and solar modulation SPhi(t).

The Be-10 scaling factor at each time step is:
SF(t) = (P10n(h, Rc(t), SPhi(t)) + P10p(h, Rc(t), SPhi(t))) / BeRef

Where BeRef = P10nRef + P10pRef from Lifton et al. (2014) consts_LSD.mat,
calibrated at Rc=0 GV, modern solar conditions (SPhi=624.57 MV, mean 2001-2010).

The production rate time series P(t) is then forward-integrated to give N(t),
and the measured N10 is reverse-interpolated to find the exposure age.

References:
- Lifton et al. (2014) EPSL 386, 149-160
- Sato et al. (2008) Radiation Research 170, 244-259
- Borchers et al. (2016) Quaternary Geochronology 31, 188-198 (P10_LSDn = 3.92)

In [82]:
def age_lsdn(N10, delN10, pressure, LSDRc, SPhi_t, tv_lsd,
             SF_thick, SF_snow, shieldcorr,
             P10_LSDn=P10_LSDn, l10=l10, delP_frac=0.37/3.92,
             w=0.066):
    """
    Exposure age from 10Be using LSDn scaling (Lifton et al. 2014).
    Forward integration + reverse interpolation.
    """
    c_lsd = lsd['consts']
    BeRef = float(c_lsd.P10nRef) + float(c_lsd.P10pRef)

    # Compute P10n and P10p for all time steps at once (vectorized)
    P10n_t, _ = sato_neutrons_py(pressure, LSDRc, SPhi_t, w, lsd, xsects)
    P10p_t, _ = sato_protons_py(pressure,  LSDRc, SPhi_t,    lsd, xsects)

    # LSDn scaling factor time series
    SF_t = (P10n_t + P10p_t) / BeRef

    # Site production rate time series
    P_t = P10_LSDn * SF_t * SF_thick * SF_snow * shieldcorr

    # Forward integration
    dcf = np.exp(-tv_lsd * l10)
    N_t = cumulative_trapezoid(P_t * dcf, tv_lsd, initial=0)

    if N10 >= np.max(N_t):
        return np.inf, np.inf, np.inf, np.inf

    t = float(interp1d(N_t, tv_lsd, kind='linear')(N10))

    # Error propagation
    A  = l10
    FP = (N10 * A) / (1 - np.exp(-A * t))
    dtdN = 1.0 / (FP - N10 * A)
    dtdP = -N10 / (FP**2 - N10 * A * FP)

    delt_int = np.abs(dtdN) * delN10
    delt_ext = np.abs(dtdP) * FP * delP_frac
    delt_tot = np.sqrt(delt_int**2 + delt_ext**2)

    return t, delt_int, delt_ext, delt_tot


# Calculate LSDn ages for all samples
print(f"{'Sample':<20} {'St(ka)':>8} {'Lm(ka)':>8} {'LSDn(ka)':>9} {'±int':>6} {'±ext':>6} {'±tot':>6}")
print("-"*65)

for i, row in df.iterrows():
    # St
    P_st = P10_St * SF_thick[i] * SF_snow[i] * SF_stone[i] * row.shieldcorr
    t_st, _, _, _ = age_st(row.N10_norm, row.delN10_norm, P_st)

    # Lm
    LmRc_i = compute_Rc_lm(row.lat, row.lon_360, tv, pmag07)
    t_lm, _, _, _ = age_lm(row.N10_norm, row.delN10_norm,
                            pressure[i], LmRc_i, tv,
                            SF_thick[i], SF_snow[i], row.shieldcorr)

    # LSDn
    tv_lsd, LSDRc_i, SPhi_i = build_lsdn_timevectors(row.lat, row.lon_360, pmag12, lsd)
    t_lsdn, dint, dext, dtot = age_lsdn(
        row.N10_norm, row.delN10_norm,
        pressure[i], LSDRc_i, SPhi_i, tv_lsd,
        SF_thick[i], SF_snow[i], row.shieldcorr
    )

    print(f"{row['sample']:<20} {t_st/1000:>8.2f} {t_lm/1000:>8.2f} "
          f"{t_lsdn/1000:>9.2f} {dint/1000:>6.2f} {dext/1000:>6.2f} {dtot/1000:>6.2f}")

Sample                 St(ka)   Lm(ka)  LSDn(ka)   ±int   ±ext   ±tot
-----------------------------------------------------------------
22-MBI-008-SBL1b        14.48    14.01     14.77   0.27   1.40   1.43
22-MBI-010-SBL2         15.39    14.87     15.62   0.30   1.48   1.51
22-MBI-020-SBL5b        15.43    14.91     15.65   0.43   1.48   1.54
22-MBI-045-SBL9b        12.69    12.28     12.78   0.24   1.21   1.23
23-MBI-009-SBL11        14.59    14.10     14.52   0.24   1.38   1.40
23-MBI-010-SBL12        14.12    13.65     14.08   0.24   1.33   1.36
22-MBI-061-SHL1         15.45    14.90     15.36   0.30   1.46   1.49
22-MBI-069-SHL2         17.17    16.48     17.03   0.32   1.61   1.65
22-MBI-076-SHL3         14.23    13.76     14.33   0.27   1.36   1.38
22-MBI-084-SHL4         15.27    14.74     15.36   0.25   1.46   1.48
22-MBI-086-SHL6         15.85    15.28     15.92   0.29   1.51   1.54
